# [Improved V6] NFL Draft Prediction

Strategy: Best of everything
- **V1 + V3 features** (all safe, leak-free)
- **Optuna hyperparameter tuning** (50 trials)
- **10-fold StratifiedKFold** (reduced variance vs 5-fold)
- **Ensemble**: average V6 predictions with V1 and V4 submission CSVs

Features used:
- Base numeric + categoricals
- BMI, missing flags, n_missing
- Position-relative z-scores
- Speed score (1/Sprint + 1/Shuttle + 1/Agility)
- Age z-score by position
- Interaction features: Height×Weight, Sprint×Vertical, Sprint×Broad_Jump

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\YSS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the Data

In [2]:
train = pd.read_csv('input/train.csv')
test  = pd.read_csv('input/test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)

Train shape: (2781, 16)
Test shape:  (696, 15)


## 3. Feature Engineering (V1 + V3, leak-free)

In [3]:
perf_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
             'Broad_Jump', 'Agility_3cone', 'Shuttle']

def engineer_features(df):
    df = df.copy()

    # V1 features
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    for col in perf_cols:
        df[f'{col}_missing'] = df[col].isna().astype(int)
    df['n_missing'] = df[perf_cols].isna().sum(axis=1)
    for col in perf_cols:
        grp = df.groupby('Position')[col]
        df[f'{col}_pos_zscore'] = (df[col] - grp.transform('mean')) / (grp.transform('std') + 1e-6)

    # V3 safe extras
    df['speed_score'] = (1.0 / (df['Sprint_40yd']  + 1e-6) +
                         1.0 / (df['Shuttle']       + 1e-6) +
                         1.0 / (df['Agility_3cone'] + 1e-6))
    age_grp = df.groupby('Position')['Age']
    df['age_pos_zscore']    = (df['Age'] - age_grp.transform('mean')) / (age_grp.transform('std') + 1e-6)
    df['height_x_weight']   = df['Height'] * df['Weight']
    df['sprint_x_vertical'] = df['Sprint_40yd'] * df['Vertical_Jump']
    df['sprint_x_broad']    = df['Sprint_40yd'] * df['Broad_Jump']

    return df

train = engineer_features(train)
test  = engineer_features(test)
print('Feature engineering done. Train shape:', train.shape)

Feature engineering done. Train shape: (2781, 35)


## 4. Preprocessing

In [4]:
cat_cols = ['Player_Type', 'Position_Type', 'Position']
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

base_num = ['Year', 'Age', 'Height', 'Weight',
            'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
            'Broad_Jump', 'Agility_3cone', 'Shuttle']

engineered = (
    ['BMI', 'n_missing'] +
    [f'{c}_missing'    for c in perf_cols] +
    [f'{c}_pos_zscore' for c in perf_cols] +
    ['speed_score', 'age_pos_zscore',
     'height_x_weight', 'sprint_x_vertical', 'sprint_x_broad']
)

features = base_num + cat_cols + engineered

X      = train[features]
y      = train['Drafted']
X_test = test[features]

print(f'Total features: {len(features)}')

Total features: 32


## 5. Hyperparameter Tuning with Optuna (10-fold)

In [5]:
def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'auc',
        'verbose':           -1,
        'n_jobs':            -1,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1':         trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
    }

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
            params, dtrain,
            num_boost_round=1000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        oof[val_idx] = model.predict(X_val)

    return roc_auc_score(y, oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest OOF AUC: {study.best_value:.5f}')
print('Best params:', study.best_params)

Best trial: 35. Best value: 0.830959: 100%|██████████| 50/50 [03:21<00:00,  4.02s/it]


Best OOF AUC: 0.83096
Best params: {'learning_rate': 0.07090148202843853, 'num_leaves': 49, 'min_child_samples': 39, 'feature_fraction': 0.9737452217554499, 'bagging_fraction': 0.5504338743733866, 'bagging_freq': 2, 'lambda_l1': 1.358335469664263e-07, 'lambda_l2': 6.465818788187726e-08}


## 6. Train Final Model with Best Params (10-fold)

In [6]:
best_params = {
    'objective': 'binary',
    'metric':    'auc',
    'verbose':   -1,
    'n_jobs':    -1,
    **study.best_params
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        best_params, dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(X_test) / 10

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f'Fold {fold+1:2d}  AUC: {fold_auc:.5f}')

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nOverall OOF AUC: {oof_auc:.5f}')

Fold  1  AUC: 0.80578
Fold  2  AUC: 0.82896
Fold  3  AUC: 0.84684
Fold  4  AUC: 0.86973
Fold  5  AUC: 0.86930
Fold  6  AUC: 0.85437
Fold  7  AUC: 0.77460
Fold  8  AUC: 0.81859
Fold  9  AUC: 0.81740
Fold 10  AUC: 0.88520

Overall OOF AUC: 0.83096


## 7. Save V6 Submission

In [7]:
submission = pd.read_csv('input/sample_submission.csv')
submission['Drafted'] = test_preds
submission.to_csv('submission_V6.csv', index=False)
print('submission_V6.csv saved!')
submission.head()

submission_V6.csv saved!


,Id,Drafted
0,2781,0.649515
1,2782,0.819793
2,2783,0.868598
3,2784,0.921577
4,2785,0.771552


## 8. Ensemble: Average V6 + V1 + V4

In [8]:
sub_v1 = pd.read_csv('submission_V1.csv')
sub_v4 = pd.read_csv('submission_V4.csv')
sub_v6 = pd.read_csv('submission_V6.csv')

ensemble = sub_v6.copy()
ensemble['Drafted'] = (sub_v1['Drafted'] + sub_v4['Drafted'] + sub_v6['Drafted']) / 3

ensemble.to_csv('submission_V6_ensemble.csv', index=False)
print('submission_V6_ensemble.csv saved!')
print(f'V1 mean:       {sub_v1["Drafted"].mean():.4f}')
print(f'V4 mean:       {sub_v4["Drafted"].mean():.4f}')
print(f'V6 mean:       {sub_v6["Drafted"].mean():.4f}')
print(f'Ensemble mean: {ensemble["Drafted"].mean():.4f}')
ensemble.head()

submission_V6_ensemble.csv saved!
V1 mean:       0.6453
V4 mean:       0.6404
V6 mean:       0.6351
Ensemble mean: 0.6403


,Id,Drafted
0,2781,0.649378
1,2782,0.815117
2,2783,0.814310
3,2784,0.867292
4,2785,0.748055
